# FinOps Cost Anomaly Benchmark

Refactored driver notebook. The experiment logic lives in `finops_benchmark/` modules.

In [ ]:
# Colab setup
!pip install -q -r requirements.txt

import os
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


In [ ]:
from finops_benchmark.config import (
    RANDOM_SEED, YEAR2_START, SEEDS, BUDGETS, METRIC_COLS, EVENT_KEEP_COLS,
    FIG_DIR, FIGS_DIR_NEW, RESULTS_DIR, ensure_output_dirs, ensure_results_dirs,
)
from finops_benchmark.data import build_dataset
from finops_benchmark.models import run_all_models
from finops_benchmark.evaluation import compute_thresholds, run_evaluation
from finops_benchmark.experiment import (
    build_core_results_table, build_rank_comparison, build_rank_table,
    build_summary_metrics, run_one_seed,
)
from finops_benchmark.visualization import (
    configure_paper_matplotlib, get_canonical_model_order, plot_anomaly_examples,
    plot_data_overview, plot_metric_by_budget, plot_paper_bar, plot_paper_line,
    plot_rank_comparison, plot_score_overview, plot_budget1_bar,
)
from finops_benchmark.sanity import run_sanity_checks

OUTPUT_DIR, FIG_DIR = ensure_output_dirs()
RESULTS_DIR, FIGS_DIR_NEW = ensure_results_dirs()
print("Config loaded.")


## Part 1. Synthetic Data

In [ ]:
df, events_df = build_dataset(seed=RANDOM_SEED)

print("=== df head ===")
print(df.head())
print("\n=== df shape ===", df.shape)
print("\n=== anomaly day count ===")
print(df.groupby(["anomaly_type", "intensity_level"]).size())
print("\n=== events_df ===")
print(events_df)
print("\n=== events_df summary ===")
print(events_df.groupby(["anomaly_type", "intensity_level"]).agg(
    n_events=("event_id", "count"),
    mean_total_excess=("total_excess_cost", "mean"),
    sum_total_excess=("total_excess_cost", "sum"),
).round(2))


In [ ]:
_ = plot_data_overview(df, events_df, save_path=os.path.join(FIG_DIR, "01_data_overview.png"))
plt.show()

_ = plot_anomaly_examples(df, events_df, save_path=os.path.join(FIG_DIR, "02_anomaly_examples.png"))
plt.show()


## Part 2. Model Scores

In [ ]:
scores_long, model_aux = run_all_models(df, events_df, seed=RANDOM_SEED, verbose=True)
print("\nscores_long shape:", scores_long.shape)
print(scores_long.groupby("model_name")["score"].describe().round(3))

_ = plot_score_overview(df, scores_long, save_path=os.path.join(FIG_DIR, "03_score_overview.png"))
plt.show()


## Part 3. Single-Seed Evaluation

In [ ]:
thresholds = compute_thresholds(scores_long, year2_start=YEAR2_START, percentile=99.0)
print("Primary thresholds (99th pct of Year 1 scores):")
for k, v in thresholds.items():
    print(f"  {k:18s} {v:.6f}")

predictions_df, model_metrics_df, event_results_df = run_evaluation(
    df, events_df, scores_long, year2_start=YEAR2_START, percentile=99.0,
)

print("=== model_metrics_df ===")
print(model_metrics_df.round(4).to_string(index=False))
print("\n=== event_results_df: detection rate by (anomaly_type, intensity_level) ===")
print(event_results_df.groupby(["model_name", "anomaly_type", "intensity_level"])["detected"].mean().unstack("intensity_level").round(2))


## Part 4. Multi-Seed Experiment

In [ ]:
all_metrics_parts = []
all_events_parts = []

for s in tqdm(SEEDS, desc="seeds"):
    m_s, e_s = run_one_seed(seed=s, verbose=False)
    all_metrics_parts.append(m_s)
    all_events_parts.append(e_s)

all_model_metrics_df = pd.concat(all_metrics_parts, axis=0, ignore_index=True)
all_event_results_df_full = pd.concat(all_events_parts, axis=0, ignore_index=True)
all_event_results_df = all_event_results_df_full[EVENT_KEEP_COLS].copy()
summary_metrics_df = build_summary_metrics(all_model_metrics_df, METRIC_COLS)
rank_table = build_rank_table(all_model_metrics_df, budget=0.01)

print("all_model_metrics_df shape:", all_model_metrics_df.shape)
print("all_event_results_df shape:", all_event_results_df.shape)
print("=== summary_metrics_df (key columns) ===")
print(summary_metrics_df[[
    "budget", "model_name", "f1_mean", "f1_std",
    "cost_weighted_recall_mean", "cost_weighted_recall_std",
    "alert_cost_efficiency_mean", "alert_cost_efficiency_std",
    "mean_mctd_mean", "mean_mctd_std",
    "false_alarm_rate_mean", "false_alarm_rate_std",
]].round(4).to_string(index=False))


## Part 5. Figures and CSV Outputs

In [ ]:
_ = plot_budget1_bar(summary_metrics_df, "f1", os.path.join(FIGS_DIR_NEW, "10_f1_bar_budget1pct.png"))
plt.show()
_ = plot_budget1_bar(summary_metrics_df, "cost_weighted_recall", os.path.join(FIGS_DIR_NEW, "11_dollar_recall_bar_budget1pct.png"))
plt.show()
_ = plot_budget1_bar(summary_metrics_df, "alert_cost_efficiency", os.path.join(FIGS_DIR_NEW, "12_ace_bar_budget1pct.png"))
plt.show()
_ = plot_metric_by_budget(summary_metrics_df, "f1", os.path.join(FIGS_DIR_NEW, "13_f1_by_budget.png"))
plt.show()
_ = plot_metric_by_budget(summary_metrics_df, "alert_cost_efficiency", os.path.join(FIGS_DIR_NEW, "14_ace_by_budget.png"))
plt.show()
_ = plot_rank_comparison(rank_table, os.path.join(FIGS_DIR_NEW, "15_rank_compare_f1_vs_ace.png"))
plt.show()

all_model_metrics_df.to_csv(os.path.join(RESULTS_DIR, "all_model_metrics.csv"), index=False)
summary_metrics_df.to_csv(os.path.join(RESULTS_DIR, "summary_metrics.csv"), index=False)
all_event_results_df.to_csv(os.path.join(RESULTS_DIR, "all_event_results.csv"), index=False)


## Part 6. Paper Tables and Figures

In [ ]:
core_results_df = build_core_results_table(summary_metrics_df, budget=0.01)
rank_comparison_df = build_rank_comparison(all_model_metrics_df, budget=0.01)

core_results_df.to_csv(os.path.join(RESULTS_DIR, "paper_core_results_budget1pct.csv"), index=False)
rank_comparison_df.to_csv(os.path.join(RESULTS_DIR, "paper_rank_comparison_budget1pct.csv"), index=False)

print("=== Core results table at budget=1% (mean +/- std over seeds) ===")
print(core_results_df.to_string(index=False))
print("\n=== Rank comparison at budget=1% ===")
print(rank_comparison_df.to_string(index=False))


In [ ]:
configure_paper_matplotlib()
CANONICAL_ORDER = get_canonical_model_order(summary_metrics_df, budget=0.01)

_ = plot_paper_bar(summary_metrics_df, "f1", os.path.join(FIGS_DIR_NEW, "paper_f1_bar_budget1pct.png"),
                   ylabel="F1 score", title=f"F1 at budget=1% (mean +/- std over {len(SEEDS)} seeds)",
                   model_order=CANONICAL_ORDER)
plt.show()
_ = plot_paper_bar(summary_metrics_df, "cost_weighted_recall", os.path.join(FIGS_DIR_NEW, "paper_dollar_recall_bar_budget1pct.png"),
                   ylabel="$-Recall (cost-weighted recall)", title=f"Cost-weighted recall at budget=1% (mean +/- std over {len(SEEDS)} seeds)",
                   model_order=CANONICAL_ORDER)
plt.show()
_ = plot_paper_bar(summary_metrics_df, "alert_cost_efficiency", os.path.join(FIGS_DIR_NEW, "paper_ace_bar_budget1pct.png"),
                   ylabel="Alert Cost Efficiency ($ per alert)", title=f"Alert Cost Efficiency at budget=1% (mean +/- std over {len(SEEDS)} seeds)",
                   model_order=CANONICAL_ORDER)
plt.show()
_ = plot_paper_line(summary_metrics_df, "f1", os.path.join(FIGS_DIR_NEW, "paper_f1_by_budget.png"),
                    ylabel="F1 score", title="F1 across alert budgets")
plt.show()
_ = plot_paper_line(summary_metrics_df, "mean_mctd", os.path.join(FIGS_DIR_NEW, "paper_mctd_by_budget.png"),
                    ylabel="Mean MCTD ($)", title="Mean Cost-To-Detect across alert budgets", lower_is_better=True)
plt.show()
_ = plot_paper_line(summary_metrics_df, "false_alarm_rate", os.path.join(FIGS_DIR_NEW, "paper_far_by_budget.png"),
                    ylabel="False Alarm Rate", title="False Alarm Rate across alert budgets", lower_is_better=True)
plt.show()


## Sanity Checks

In [ ]:
sanity_summary, sanity_details = run_sanity_checks(
    df=df,
    events_df=events_df,
    scores_long=scores_long,
    thresholds=thresholds,
    predictions_df=predictions_df,
    model_metrics_df=model_metrics_df,
    all_model_metrics_df=all_model_metrics_df,
    all_event_results_df=all_event_results_df,
)

print("=" * 88)
print("SANITY CHECK SUMMARY")
print("=" * 88)
with pd.option_context("display.max_colwidth", 110):
    print(sanity_summary.to_string(index=False))

print("\nAUPRC verification detail (Check 8)")
print(sanity_details["auprc_table"].round(6).to_string(index=False))
